# Text-to-Music Generation

## Task Description
Generate music from a text description.

**Focus:** This notebook uses the `MusicGen` model from Meta (Facebook). We will use the `small` version which fits comfortably on the Google Colab Free Tier (T4 GPU).

In this notebook, we will:
1. **Install:** Install `transformers` and audio libraries.
2. **Load Model:** Load `facebook/musicgen-small`.
3. **Inference:** Generate music from a text prompt.
4. **Save/Play:** Save the output as a `.wav` file and play it.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## Learning Objectives

*   **Practical:** Gain hands-on experience with this model.
*   **Theory:** Understand the underlying concepts and architecture.

In [ ]:
# Check for GPU
!nvidia-smi

## 1. Installation
We need `transformers` and `scipy` to save the audio.

In [ ]:
!pip install matplotlib -q transformers scipy torch

## 2. Imports
Import necessary libraries.

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch
import scipy.io.wavfile

device = "cuda" if torch.cuda.is_available() else "cpu"



## 3. Load Data
Define the text prompt for music generation.

In [ ]:
categories = {
    "Pop": "80s pop track with bassy drums and synth",
    "Rock": "Upbeat rock song with electric guitar",
    "Cinematic": "Epic orchestral score for a movie scene",
    "Meditation (Alpha)": "Ambient music with 10Hz binaural beats for Alpha brainwave entrainment",
    "Meditation (Theta)": "Deep meditation music with 6Hz Theta waves",
    "Meditation (Delta)": "Healing sleep music with 2Hz Delta waves",
    "Meditation (Gamma)": "Focus music with 40Hz Gamma waves"
}
category = "Meditation (Theta)" # Default
prompt = categories[category]
print(f"Category: {category}")
print(f"Prompt: {prompt}")

## 4. Data Cleaning
Clean and prepare the input prompt.

In [ ]:
prompt = prompt.strip()
print("Prompt cleaned.")

## 5. Load Model
Load the MusicGen model.

In [ ]:
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
model.to(device)

print("MusicGen model loaded.")

## 6. Inference
Generate music.

In [ ]:
inputs = processor(
    text=[prompt],
    padding=True,
    return_tensors="pt",
)

# Generate 5 seconds of audio
audio_values = model.generate(**inputs.to(device), max_new_tokens=256)

print("Audio generated.")

## 7. Save and Play
Save and listen to the audio.

In [ ]:
sampling_rate = model.config.audio_encoder.sampling_rate
audio_data = audio_values[0, 0].cpu().numpy()

scipy.io.wavfile.write("musicgen_out.wav", rate=sampling_rate, data=audio_data)
print("Saved to musicgen_out.wav")

from IPython.display import Audio
display(Audio(audio_data, rate=sampling_rate))

## 7. Save Results
Log the completion.

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage
Try using a different model or your own data.


**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
# Example: Load a different model
# model_name = "..."
# classifier = pipeline(..., model=model_name)
# results = classifier(your_input)
print("Try modifying the code above to use a different model!")

## 9. Troubleshooting

*   **Out of Memory (OOM):** If you encounter OOM errors, try reducing the batch size or input length. On the Free Tier, you are limited to T4 GPUs with 16GB VRAM.
*   **GPU Not Found:** Ensure your runtime type is set to GPU. Go to *Runtime > Change runtime type* and select *T4 GPU*.
*   **Missing Packages:** If you see `ModuleNotFoundError`, make sure you have run the **Installation** step (Step 1).